In [ ]:
# 18_recourse_fullset.ipynb
# Full-set recourse share: fraction of ALL top-decile high-risk uninsured households for
# which DiCE recommends acquiring insurance (generalizes the top-20 headline). Writes t20.
# Note: genetic DiCE over ~150 households takes several minutes.
# Full-set recourse share: fraction of ALL high-risk uninsured households for which
# DiCE ever recommends acquiring insurance (not just the top-20 used in the main text).
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
import dice_ml
PROC, TAB = "../data/processed", "../results/tables"
p=pd.read_parquet(f"{PROC}/khp_panel.parquet")
num=["age","hh_size","log_hhinc"]; cat=["female","chronic","htn","dm","disabled","HEALTH_INS","ECO1","EDU","REGION1","hh_phi_any"]
tgt="che25"; df=p[num+cat+[tgt,"PIDWON"]].dropna().copy(); df[tgt]=df[tgt].astype(int)
for c in cat: df[c]=df[c].astype(int).astype(str)
gss=GroupShuffleSplit(1,test_size=0.25,random_state=42); tr,te=next(gss.split(df,groups=df["PIDWON"]))
train,test=df.iloc[tr].copy(),df.iloc[te].copy()
pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat)],remainder="passthrough")
clf=Pipeline([("pre",pre),("gb",HistGradientBoostingClassifier(max_iter=350,learning_rate=0.06,max_depth=4,l2_regularization=1.0,random_state=0))])
clf.fit(train[num+cat],train[tgt])
unins=test[test["hh_phi_any"]=="0"].copy(); unins["risk"]=clf.predict_proba(unins[num+cat])[:,1]
thr=unins["risk"].quantile(0.90)
hi=unins[unins["risk"]>=thr][num+cat].reset_index(drop=True)
print(f"high-risk uninsured (top risk decile): n={len(hi)}; risk threshold={thr:.3f}")
N=min(len(hi),150); hiN=hi.head(N)
dd=dice_ml.Data(dataframe=train[num+cat+[tgt]],continuous_features=num,outcome_name=tgt)
m=dice_ml.Model(model=clf,backend="sklearn")
exp=dice_ml.Dice(dd,m,method="genetic")
vary=["hh_phi_any","HEALTH_INS","ECO1","log_hhinc"]
cf=exp.generate_counterfactuals(hiN,total_CFs=2,desired_class=0,features_to_vary=vary)
valid=phi_hh=ncf=0
for i in range(len(hiN)):
    c=cf.cf_examples_list[i].final_cfs_df
    if c is None or len(c)==0: continue
    valid+=1; o=hiN.iloc[i]; rec=False
    for j in range(len(c)):
        ncf+=1
        if str(c.iloc[j]["hh_phi_any"])!=str(o["hh_phi_any"]): phi_hh+=1; rec=True
print(f"evaluated N={N}; valid recourse for {valid}; counterfactuals={ncf}")
print(f"PHI acquisition appears in {phi_hh} of {ncf} counterfactuals ({100*phi_hh/max(ncf,1):.1f}%)")
pd.DataFrame([{"n_highrisk_uninsured":len(hi),"evaluated":N,"valid":valid,"n_cf":ncf,
               "phi_recommend_pct":100*phi_hh/max(ncf,1)}]).to_csv(f"{TAB}/t20_recourse_fullset.csv",index=False)
